#Coffee Shop Sales Analysis

In [ ]:
import pandas as pd
import numpy as np

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
#Dataset:
df = pd.read_csv("Coffee_Shop_Sales.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Rename common alternative column names
df = df.rename(columns={
    "date": "transaction_date",
    "quantity": "transaction_qty",
    "qty": "transaction_qty",
    "price": "unit_price",
    "category": "product_category",
    "product": "product_detail",
    "branch": "store_location",
    "customer": "customer_id"
})

# Convert date and numerical columns
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"], errors="coerce"
)

df["transaction_qty"] = pd.to_numeric(
    df["transaction_qty"], errors="coerce"
)

df["unit_price"] = pd.to_numeric(
    df["unit_price"], errors="coerce"
)

# Remove rows with missing essential values
df = df.dropna(
    subset=["transaction_date", "transaction_qty", "unit_price"]
)

# Calculate sales revenue
df["sales"] = df["transaction_qty"] * df["unit_price"]

# Create time-related columns
df["month"] = df["transaction_date"].dt.month_name()
df["weekday"] = df["transaction_date"].dt.day_name()
df["year_month"] = df["transaction_date"].dt.to_period("M").astype(str)

print("COFFEE SHOP SALES ANALYSIS")
print("Total records:", len(df))

COFFEE SHOP SALES ANALYSIS
Total records: 20


In [ ]:
# Q1: Overall Sales:

total_sales = np.sum(df["sales"])

monthly_sales = df.groupby("year_month")["sales"].sum()

print("\nQ1: OVERALL SALES")
print("Total Sales: ₹", round(total_sales, 2))
print("\nMonthly Sales:")
print(monthly_sales)


Q1: OVERALL SALES
Total Sales: ₹ 8100

Monthly Sales:
year_month
2026-01    8100
Name: sales, dtype: int64


In [ ]:
# Q2: Categories and products with largest and smallest revenue

category_sales = df.groupby("product_category")["sales"].sum()
product_sales = df.groupby("product_detail")["sales"].sum()

print("\nQ2: CATEGORY SALES")
print(category_sales.sort_values(ascending=False))

print("\nHighest Revenue Category:", category_sales.idxmax())
print("Lowest Revenue Category:", category_sales.idxmin())

print("\nQ2: PRODUCT SALES")
print(product_sales.sort_values(ascending=False))

print("Highest Revenue Product:", product_sales.idxmax())
print("Lowest Revenue Product:", product_sales.idxmin())


Q2: CATEGORY SALES
product_category
Coffee     5450
Dessert    1440
Bakery      810
Tea         400
Name: sales, dtype: int64

Highest Revenue Category: Coffee
Lowest Revenue Category: Tea

Q2: PRODUCT SALES
product_detail
Latte          1800
Cold Coffee    1600
Brownie        1440
Cappuccino     1050
Espresso       1000
Croissant       810
Green Tea       400
Name: sales, dtype: int64
Highest Revenue Product: Latte
Lowest Revenue Product: Green Tea


In [ ]:
# Q3: Branch performance

branch_sales = df.groupby("store_location")["sales"].sum()

print("\nQ3: BRANCH PERFORMANCE")
print(branch_sales.sort_values(ascending=False))

print("Best Performing Branch:", branch_sales.idxmax())
print("Branch Needing Investigation:", branch_sales.idxmin())


Q3: BRANCH PERFORMANCE
store_location
B    4390
A    3710
Name: sales, dtype: int64
Best Performing Branch: B
Branch Needing Investigation: A


In [ ]:
# Q4: Monthly, weekday and time-period sales

weekday_sales = df.groupby("weekday")["sales"].sum()

# Arrange weekdays in calendar order
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_sales = weekday_sales.reindex(weekday_order)

print("\nQ4: WEEKDAY SALES")
print(weekday_sales)

# Identify stronger and weaker weekdays
print("Best Weekday:", weekday_sales.idxmax())
print("Weakest Weekday:", weekday_sales.idxmin())

# Identify time periods if transaction_time exists
if "transaction_time" in df.columns:

    df["hour"] = pd.to_datetime(
        df["transaction_time"], format="%H:%M:%S",
        errors="coerce"
    ).dt.hour

    if df["hour"].isna().all():
        df["hour"] = pd.to_datetime(
            df["transaction_time"], errors="coerce"
        ).dt.hour

    df["period"] = pd.cut(
        df["hour"],
        bins=[-1, 11, 15, 18, 23],
        labels=["Morning", "Afternoon", "Evening", "Night"]
    )

    period_sales = df.groupby(
        "period", observed=False
    )["sales"].sum()

    print("\nSales by Time Period:")
    print(period_sales)

else:
    print("\nTransaction time data is unavailable.")


Q4: WEEKDAY SALES
weekday
Monday        960
Tuesday      1740
Wednesday     810
Thursday      960
Friday       1250
Saturday     1100
Sunday       1280
Name: sales, dtype: int64
Best Weekday: Tuesday
Weakest Weekday: Wednesday

Transaction time data is unavailable.


In [ ]:
# Q5: Average transaction value by category and location

transaction_sales = df.groupby("transaction_id")["sales"].sum()

average_transaction_value = np.mean(transaction_sales)

print("\nQ5: AVERAGE TRANSACTION VALUE")
print("Overall Average: ₹", round(average_transaction_value, 2))

if "transaction_id" in df.columns:

    category_transaction = df.groupby(
        ["product_category", "transaction_id"]
    )["sales"].sum()

    category_average = category_transaction.groupby(
        level=0
    ).mean()

    branch_transaction = df.groupby(
        ["store_location", "transaction_id"]
    )["sales"].sum()

    branch_average = branch_transaction.groupby(
        level=0
    ).mean()

    print("\nAverage Transaction Value by Category:")
    print(category_average)

    print("\nAverage Transaction Value by Branch:")
    print(branch_average)


Q5: AVERAGE TRANSACTION VALUE
Overall Average: ₹ 405.0

Average Transaction Value by Category:
product_category
Bakery     405.000000
Coffee     419.230769
Dessert    480.000000
Tea        200.000000
Name: sales, dtype: float64

Average Transaction Value by Branch:
store_location
A    371.0
B    439.0
Name: sales, dtype: float64


In [ ]:
# Q6: Relationship between discounts, quantity and transaction value

discount_columns = [
    "discount", "discount_amount", "discount_percentage"
]

available_discount = [
    col for col in discount_columns if col in df.columns
]

if available_discount:

    discount_column = available_discount[0]

    discount_analysis = df[
        [discount_column, "transaction_qty", "sales"]
    ].corr()

    print("\nQ6: DISCOUNT ANALYSIS")
    print(discount_analysis)

    print("Correlation between discount and quantity:",
          df[discount_column].corr(df["transaction_qty"]))

    print("Correlation between discount and sales:",
          df[discount_column].corr(df["sales"]))

else:
    print("\nQ6: Discount information is not available.")
    print("This question cannot be answered using this dataset.")


Q6: DISCOUNT ANALYSIS
                     discount_percentage  transaction_qty     sales
discount_percentage             1.000000         0.307729  0.523492
transaction_qty                 0.307729         1.000000  0.564399
sales                           0.523492         0.564399  1.000000
Correlation between discount and quantity: 0.3077287274483318
Correlation between discount and sales: 0.5234918863615835


In [ ]:
# Q7: Frequent and high-value customers

if "customer_id" in df.columns:

    customer_analysis = df.groupby("customer_id").agg(
        total_purchases=("transaction_id", "nunique"),
        total_spending=("sales", "sum")
    )

    print("\nQ7: TOP FREQUENT CUSTOMERS")
    print(customer_analysis.sort_values(
        "total_purchases", ascending=False
    ).head(10))

    print("\nQ7: TOP HIGH-VALUE CUSTOMERS")
    print(customer_analysis.sort_values(
        "total_spending", ascending=False
    ).head(10))

else:
    print("\nQ7: Customer ID is not available.")
    print("Individual customer behaviour cannot be analysed.")


Q7: TOP FREQUENT CUSTOMERS
             total_purchases  total_spending
customer_id                                 
C01                        5            2000
C02                        4            1750
C03                        3            1100
C04                        2             800
C05                        2             750
C06                        2             900
C07                        2             800

Q7: TOP HIGH-VALUE CUSTOMERS
             total_purchases  total_spending
customer_id                                 
C01                        5            2000
C02                        4            1750
C03                        3            1100
C06                        2             900
C04                        2             800
C07                        2             800
C05                        2             750


In [ ]:
# Q8: Business recommendations

print("\nQ8: BUSINESS RECOMMENDATIONS")

print("1. Promote products and categories with lower sales.")
print("2. Maintain sufficient stock of high-revenue products.")
print("3. Investigate branches with comparatively lower revenue.")
print("4. Plan staff and stock according to busy weekdays and periods.")
print("5. Encourage repeat purchases through loyalty programmes.")
print("6. Improve customer experience instead of depending only on discounts.")


# GRAPH-READY DATA
# These tables can be used later to create graphs.

print("\nGRAPH-READY DATA")

print("\nMonthly Sales Data:")
print(monthly_sales)

print("\nCategory Sales Data:")
print(category_sales)

print("\nProduct Sales Data:")
print(product_sales)

print("\nBranch Sales Data:")
print(branch_sales)

print("\nWeekday Sales Data:")
print(weekday_sales)

print("\nAnalysis Completed Successfully!")


Q8: BUSINESS RECOMMENDATIONS
1. Promote products and categories with lower sales.
2. Maintain sufficient stock of high-revenue products.
3. Investigate branches with comparatively lower revenue.
4. Plan staff and stock according to busy weekdays and periods.
5. Encourage repeat purchases through loyalty programmes.
6. Improve customer experience instead of depending only on discounts.

GRAPH-READY DATA

Monthly Sales Data:
year_month
2026-01    8100
Name: sales, dtype: int64

Category Sales Data:
product_category
Bakery      810
Coffee     5450
Dessert    1440
Tea         400
Name: sales, dtype: int64

Product Sales Data:
product_detail
Brownie        1440
Cappuccino     1050
Cold Coffee    1600
Croissant       810
Espresso       1000
Green Tea       400
Latte          1800
Name: sales, dtype: int64

Branch Sales Data:
store_location
A    3710
B    4390
Name: sales, dtype: int64

Weekday Sales Data:
weekday
Monday        960
Tuesday      1740
Wednesday     810
Thursday      960
Friday